# V024 synthetic and runtime proof

No approved data are read. Tests establish exact identity initialization, convolution outputs/gradients, protected known inputs, independent breaths, a zero-adapter training control, exact interrupted optimizer/model/RNG recovery, native validation/inference, and rejection of changed span/source. Actual CUDA also checks the full deep backbone and bounded adapter overhead before any approved-data fit.

In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
from pathlib import Path
from datetime import datetime,timezone
import json,hashlib,time,statistics
import numpy as np
import torch

def run_convolution_preflight_v024(root,output_directory,device='cpu',source_fingerprints=None,deep_spec=None):
    root=Path(root).resolve();O=Path(output_directory);O.mkdir(parents=True,exist_ok=False)
    if device not in ('cpu','cuda'):raise ValueError('Explicit CPU or actual CUDA proof required')
    if device=='cuda' and not torch.cuda.is_available():raise RuntimeError('Actual CUDA required')
    def code(p):return '\n'.join(''.join(c['source']) for c in json.loads(Path(p).read_text())['cells'] if c['cell_type']=='code')
    sources={} if source_fingerprints is None else dict(source_fingerprints)
    ns={'__name__':'ventilator_v024_components'}
    for name in ['feature_components_v005.ipynb','sequence_components_v005.ipynb','training_components_v008.ipynb','convolution_components_v024.ipynb']:
        s=code(root/'research'/name);h=hashlib.sha256(s.encode()).hexdigest()
        if name in sources and sources[name]!=h:raise ValueError('Pinned source changed: '+name)
        sources[name]=h;exec(compile(s,name,'exec'),ns)
    sources['convolution_preflight_v024.ipynb']=hashlib.sha256(code(root/'research/convolution_preflight_v024.ipynb').encode()).hexdigest()
    torch.set_num_threads(1 if device=='cpu' else 4)
    checks=ns['convolution_selfcheck_v024'](device)
    def check(k,v):
        if not bool(v):raise AssertionError(k)
        checks[k]=True
    def equal(a,b):
        if isinstance(a,torch.Tensor):return torch.equal(a,b)
        if isinstance(a,np.ndarray):return np.array_equal(a,b)
        if isinstance(a,dict):return a.keys()==b.keys() and all(equal(v,b[k]) for k,v in a.items())
        if isinstance(a,(tuple,list)):return len(a)==len(b) and all(equal(x,y) for x,y in zip(a,b))
        return a==b
    rng=np.random.default_rng(20262402);names=['R','C','u_in','u_out','time_step','step','delta_u_in']
    raw=rng.normal(size=(8,80,len(names))).astype(np.float32);raw[:,:,0]=5;raw[:,:,1]=10;raw[:,:,3]=0;raw[:,45:,3]=1
    raw[:,:,4]=np.arange(80)*.03;raw[:,:,5]=np.arange(80)
    target=(10+.05*np.cumsum(raw[:,:,2],axis=1)).astype(np.float32);phase=raw[:,:,3].astype(np.int8)
    spec=dict(family='lstm',hidden=4,layers=2,dropout=.15)
    cfg=dict(lr=.001,batch_size=3,max_epochs=3,patience=0,scheduler='constant',prediction_batch_size=3,loss='l1');rates=[.001]*3
    shared=dict(feature_names=names,device=device,loss_kind='curve_l1',source_fingerprints=sources)
    ns['fit_fixed_v008'](raw[:6],target[:6],phase[:6],spec,cfg,2402,rates,O/'base.pt',model_factory=ns['v24_plain_factory'],arm='control',**shared)
    ns['fit_fixed_v008'](raw[:6],target[:6],phase[:6],spec,cfg,2402,rates,O/'frozen.pt',model_factory=ns['v24_frozen_conv3_factory'],arm='frozen',**shared)
    base=torch.load(O/'base.pt',map_location='cpu',weights_only=False);frozen=torch.load(O/'frozen.pt',map_location='cpu',weights_only=False)
    check('zero_frozen_adapter_exact_curve_training_control',all(torch.equal(v,frozen['state_dict']['backbone.'+k]) for k,v in base['state_dict'].items()))
    full_models=[]
    for span in (3,7):
        factory=ns['v24_conv'+str(span)+'_factory'];arm='preflight_conv'+str(span)
        def fit(path,**kwargs):
            return ns['fit_fixed_v008'](raw[:6],target[:6],phase[:6],spec,cfg,2402,rates,path,model_factory=factory,arm=arm,**shared,**kwargs)
        fit(O/('full'+str(span)+'.pt'))
        def stop(record):
            if record['epoch']==1:raise RuntimeError('controlled interruption')
        try:fit(O/('resume'+str(span)+'.pt'),on_epoch=stop)
        except RuntimeError as exc:check(str(span)+':controlled_interrupt_saved',str(exc)=='controlled interruption')
        else:raise AssertionError('Missing controlled interruption')
        fit(O/('resume'+str(span)+'.pt'),resume=True)
        full=torch.load(O/('full'+str(span)+'.pt'),map_location='cpu',weights_only=False)
        resumed=torch.load(O/('resume'+str(span)+'.pt'),map_location='cpu',weights_only=False)
        for key in ['state_dict','optimizer_state','rng_state']:
            check(str(span)+':exact_resume_'+key,equal(full[key],resumed[key]))
        check(str(span)+':adapter_updated_after_training',bool((full['state_dict']['context_weight']!=0).any()))
        pred=ns['predict_checkpoint_v008'](O/('full'+str(span)+'.pt'),raw[6:],device=device,model_factory=factory,
            expected_arm=arm,expected_loss_kind='curve_l1',source_fingerprints=sources)
        check(str(span)+':native_label_free_inference',pred.shape==(2,80) and np.isfinite(pred).all())
        full_models.append(full)
    report=ns['fit_validate_v008'](raw[:6],target[:6],phase[:6],raw[6:],target[6:],phase[6:],spec,cfg,2402,O/'validation.pt',
        feature_names=names,device=device,model_factory=ns['v24_conv7_factory'],loss_kind='curve_l1',arm='validation',source_fingerprints=sources)
    pred=ns['predict_checkpoint_v008'](O/'validation.pt',raw[6:],device=device,model_factory=ns['v24_conv7_factory'],
        expected_arm='validation',expected_loss_kind='curve_l1',source_fingerprints=sources)
    score=np.abs(pred.astype(np.float64)-target[6:].astype(np.float64))[phase[6:]==0].mean()
    check('validation_native_best_checkpoint_metric',abs(score-report['validation_mae'])<1e-15)
    check('training_only_scaler',report['preprocessing']['fitted_training_breaths']==6 and np.array_equal(np.asarray(report['preprocessing']['x_mean'],np.float32),raw[:6].mean((0,1),dtype=np.float64).astype(np.float32)))
    try:ns['load_checkpoint_v008'](O/'full3.pt',device=device,model_factory=ns['v24_conv7_factory'],source_fingerprints=sources)
    except ValueError:checks['changed_temporal_span_rejected']=True
    else:raise AssertionError('Changed span accepted')
    changed=dict(sources);changed['convolution_components_v024.ipynb']='0'*64
    try:ns['load_checkpoint_v008'](O/'full3.pt',device=device,model_factory=ns['v24_conv3_factory'],source_fingerprints=changed)
    except ValueError:checks['changed_component_source_rejected']=True
    else:raise AssertionError('Changed implementation accepted')
    profiles=[]
    if device=='cuda':
        if deep_spec is None:raise ValueError('Actual full-size model proof required')
        full_names=ns['feature_names_v005']('combined')
        for name,factory in [('baseline',ns['v24_plain_factory']),('conv3',ns['v24_conv3_factory']),('conv7',ns['v24_conv7_factory'])]:
            ns['seed_sequence'](20261102);torch.cuda.empty_cache()
            model=factory(deep_spec,47,{'feature_names':full_names}).to(device)
            xx=torch.randn(256,80,48,device=device);xx[...,-1]=0
            yy=torch.randn(256,80,device=device);mask=torch.zeros(256,80,dtype=torch.bool,device=device);mask[:,:45]=True
            def step():
                model.zero_grad(set_to_none=True);prediction=model(xx)
                loss=ns['curve_l1_loss_v008'](prediction,yy,mask)[0];loss.backward()
                return bool(torch.isfinite(loss)) and all(bool(torch.isfinite(v.grad).all()) for v in model.parameters() if v.grad is not None)
            check(name+':actual_deep_CUDA_gradients',step());torch.cuda.synchronize();torch.cuda.reset_peak_memory_stats()
            times=[]
            for _ in range(3):
                start=time.monotonic();step();torch.cuda.synchronize();times.append(time.monotonic()-start)
            profiles.append(dict(name=name,median_full_batch_step_seconds=statistics.median(times),
                peak_allocated_bytes=torch.cuda.max_memory_allocated(),parameter_count=sum(p.numel() for p in model.parameters())))
            del model,xx,yy,mask;torch.cuda.empty_cache()
        baseline=profiles[0]['median_full_batch_step_seconds']
        for row in profiles[1:]:
            row['step_time_ratio_to_baseline']=row['median_full_batch_step_seconds']/baseline
            check(row['name']+':bounded_GPU_overhead',row['step_time_ratio_to_baseline']<=1.7)
            check(row['name']+':bounded_peak_memory',row['peak_allocated_bytes']<10*1024**3)
    result=dict(status='passed',created_utc=datetime.now(timezone.utc).isoformat(),checks=checks,check_count=len(checks),
        component_code_sha256=sources['convolution_components_v024.ipynb'],source_fingerprints=sources,
        approved_data_reads=0,approved_data_fits=0,actual_cuda=device=='cuda',actual_GPU_preflight_pending=device!='cuda',
        exact_resume=all(checks[str(span)+':exact_resume_'+key] for span in (3,7) for key in ['state_dict','optimizer_state','rng_state']),
        zero_adapter_control=checks['zero_frozen_adapter_exact_curve_training_control'],
        native_inference=all(checks[str(span)+':native_label_free_inference'] for span in (3,7)),
        efficiency_profiles=profiles,runtime=dict(torch=str(torch.__version__),cuda=torch.version.cuda,numpy=np.__version__,device=device))
    if device=='cuda':result['GPU']=torch.cuda.get_device_name(0)
    path=O/'proof.json';path.write_text(json.dumps(result,indent=2)+'\n')
    return result

print('Synthetic-only V024 proof is inert until run_convolution_preflight_v024(...) is called.')
